# Инициализация

Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [19]:
import os
import boto3
from dotenv import load_dotenv
import sys
import scipy.sparse
from IPython.display import display
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


load_dotenv()

True

In [20]:
import utils
import importlib

importlib.reload(utils)

<module 'utils' from '/home/mle-user/sprint5/final_project/utils.py'>

In [28]:
import boto3, os
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)
bucket_name = os.getenv("S3_BUCKET_NAME")

In [29]:
objects = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket_name):
    for obj in page.get("Contents", []):
        objects.append({
            "key": obj["Key"],
            "size_mb": obj["Size"] / 1024**2,
            "last_modified": obj["LastModified"],
        })

objects_sorted = sorted(
    objects,
    key=lambda x: x["size_mb"],
    reverse=True
)

for obj in objects_sorted[:50]:
    print(
        f'{obj["size_mb"]:10.2f} MB | '
        f'{obj["last_modified"]} | '
        f'{obj["key"]}'
    )

    341.61 MB | 2026-09-27 18:01:01.875000+00:00 | recsys/data/items.parquet
    231.39 MB | 2026-09-18 09:28:59.544000+00:00 | recsys/recommendations/recommendations.parquet
    146.00 MB | 2026-09-27 17:59:13.048000+00:00 | recsys/recommendations/similar.parquet
     38.23 MB | 2026-09-27 18:01:02.492000+00:00 | recsys/data/events.parquet
     18.20 MB | 2026-06-11 14:12:22.639000+00:00 | files/md5/eb/9344ea0dd74cdcb5513f6de662a828
     15.52 MB | 2026-06-14 10:27:58.965000+00:00 | files/md5/d6/2e824fe2d6bfa01603e58b274094b9
     11.94 MB | 2026-08-11 14:57:11.740000+00:00 | 0/cc1d30d7c12c446d83bad99168271025/artifacts/data/step5.csv
     11.94 MB | 2026-08-11 14:37:50.031000+00:00 | step5.csv
     11.49 MB | 2026-08-12 15:24:37.623000+00:00 | 25/14e47d8c8434403ba8dd67aff7f628fb/artifacts/data/step5.csv
     11.49 MB | 2026-08-12 15:55:36.716000+00:00 | 25/704dd28e864a401883ff47bdeb71887c/artifacts/data/step5.csv
     11.49 MB | 2026-08-12 14:00:21.434000+00:00 | 25/9b6a2b64f8204acdb

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

In [4]:
# ============================================================
# ПОДГОТОВКА ДАННЫХ ДЛЯ РАНЖИРУЮЩЕЙ МОДЕЛИ
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Загружаем кандидатов с уже построенными признаками
# ------------------------------------------------------------

candidates = pd.read_parquet(
    "candidates_result.parquet"
)


# ------------------------------------------------------------
# 2. Подготавливаем events
# ------------------------------------------------------------

events = pd.read_parquet(
    "parquet_cleaned/events.parquet"
)


# Если названия ещё не приведены к общему виду
events_rank = events.rename(
    columns={
        "visitorid": "user_id",
        "itemid": "item_id",
    }
)


# ------------------------------------------------------------
# 3. Приводим timestamp к datetime
# ------------------------------------------------------------

if pd.api.types.is_numeric_dtype(
    events_rank["timestamp"]
):
    events_rank["event_time"] = pd.to_datetime(
        events_rank["timestamp"],
        unit="ms"
    )
else:
    events_rank["event_time"] = pd.to_datetime(
        events_rank["timestamp"]
    )


print(
    "Период данных:",
    events_rank["event_time"].min(),
    "->",
    events_rank["event_time"].max()
)

Период данных: 2015-05-03 03:00:04.384000 -> 2015-09-18 02:59:47.788000


In [5]:
# ============================================================
# ВРЕМЕННОЙ SPLIT ДЛЯ RANKER
# ============================================================

data_end = (
    events_rank["event_time"].max()
    + pd.Timedelta(milliseconds=1)
)

# Последние 7 дней -> финальная оценка
eval_start = (
    data_end
    - pd.Timedelta(days=7)
)

# Предыдущие 7 дней -> обучение ranker
ranker_start = (
    eval_start
    - pd.Timedelta(days=7)
)


print(
    "История для построения рекомендаций:",
    events_rank["event_time"].min(),
    "->",
    ranker_start
)

print(
    "Ranker train:",
    ranker_start,
    "->",
    eval_start
)

print(
    "Evaluation:",
    eval_start,
    "->",
    data_end
)

История для построения рекомендаций: 2015-05-03 03:00:04.384000 -> 2015-09-04 02:59:47.789000
Ranker train: 2015-09-04 02:59:47.789000 -> 2015-09-11 02:59:47.789000
Evaluation: 2015-09-11 02:59:47.789000 -> 2015-09-18 02:59:47.789000


In [6]:
# ============================================================
# 4. ФОРМИРУЕМ ПЕРИОДЫ RANKER TRAIN И EVALUATION
# ============================================================

ranker_events = events_rank[
    (events_rank["event_time"] >= ranker_start)
    &
    (events_rank["event_time"] < eval_start)
].copy()


eval_events = events_rank[
    (events_rank["event_time"] >= eval_start)
    &
    (events_rank["event_time"] < data_end)
].copy()


print(
    "Ranker train:",
    ranker_events["event_time"].min(),
    "->",
    ranker_events["event_time"].max()
)

print(
    "Evaluation:",
    eval_events["event_time"].min(),
    "->",
    eval_events["event_time"].max()
)

print(
    "Ranker events:",
    len(ranker_events)
)

print(
    "Evaluation events:",
    len(eval_events)
)

Ranker train: 2015-09-04 02:59:55.643000 -> 2015-09-11 02:59:45.585000
Evaluation: 2015-09-11 02:59:50.408000 -> 2015-09-18 02:59:47.788000
Ranker events: 132186
Evaluation events: 114738


In [7]:
# ============================================================
# 5. ФОРМИРУЕМ TARGET
# ============================================================

POSITIVE_EVENTS = [
    "addtocart",
    "transaction",
]


positive_pairs = (
    ranker_events[
        ranker_events["event"].isin(
            POSITIVE_EVENTS
        )
    ][
        [
            "user_id",
            "item_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


positive_pairs["target"] = 1


print(
    "Positive user-item pairs:",
    len(positive_pairs)
)

display(
    positive_pairs.head()
)

Positive user-item pairs: 3125


,user_id,item_id,target
1200121,685118,264476,1
1200165,237571,421931,1
1200177,981375,315939,1
1200222,685118,286268,1
1202317,703583,138454,1


In [ ]:
import duckdb

positive_pairs_count = len(positive_pairs)

positive_found = duckdb.sql("""
    SELECT COUNT(*)
    FROM positive_pairs p
    INNER JOIN read_parquet(
        'candidates_result.parquet'
    ) c
        ON p.user_id = c.user_id
       AND p.item_id = c.item_id
""").fetchone()[0]

coverage = (
    positive_found / positive_pairs_count
    if positive_pairs_count > 0
    else 0
)

print(
    "Всего positive pairs:",
    positive_pairs_count
)

print(
    "Positive найдено среди candidates:",
    positive_found
)

print(
    "Coverage:",
    coverage
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Всего positive pairs: 3125
Positive найдено среди candidates: 126
Coverage: 0.04032


In [9]:
FEATURES = [
    "als_score",
    "als_rank",
    "similarity_score",
    "similarity_rank",
    "popular_score",
    "popular_rank",
]

In [10]:
candidates.head(1)

,user_id,item_id,als_score,als_rank,popular_score,popular_rank,similarity_score,similarity_rank,similarity_sources,from_als,from_popular,from_similar,popular_score_log
0,1228945,450082,0.00211,44.0,33.0,51.0,NaN,NaN,NaN,1,1,0,3.526361


In [11]:
missing_features = [
    feature
    for feature in FEATURES
    if feature not in candidates.columns
]

if missing_features:
    raise ValueError(
        f"В candidates отсутствуют признаки: {missing_features}"
    )

In [12]:
# ============================================================
# СОЗДАЁМ ОБУЧАЮЩУЮ ТАБЛИЦУ RANKER
# ============================================================

ranker_users = positive_pairs[
    "user_id"
].unique()


# Сначала сокращаем огромный candidates
ranker_train = candidates[
    candidates["user_id"].isin(
        ranker_users
    )
].copy()


# Добавляем target
ranker_train = ranker_train.merge(
    positive_pairs,
    on=[
        "user_id",
        "item_id",
    ],
    how="left",
    validate="many_to_one",
)


ranker_train["target"] = (
    ranker_train["target"]
    .fillna(0)
    .astype("int8")
)


print(
    "Ranker rows:",
    len(ranker_train)
)

print(
    "Positive:",
    ranker_train["target"].sum()
)

print(
    "Positive share:",
    ranker_train["target"].mean()
)

Ranker rows: 80462
Positive: 126
Positive share: 0.001565956600631354


In [13]:

group_stats = (
    ranker_train
    .groupby("user_id")["target"]
    .agg(
        positives="sum",
        total="count"
    )
)





group_stats["negatives"] = (
    group_stats["total"]
    - group_stats["positives"]
)


valid_users = group_stats[
    (group_stats["positives"] > 0)
    &
    (group_stats["negatives"] > 0)
].index


ranker_train = ranker_train[
    ranker_train["user_id"].isin(
        valid_users
    )
].copy()




print(
    "Пользователей:",
    ranker_train["user_id"].nunique()
)

print(
    "Строк:",
    len(ranker_train)
)

print(
    "Доля target=1:",
    ranker_train["target"].mean()
)

Пользователей: 95
Строк: 29494
Доля target=1: 0.0042720553332881266


In [14]:
from catboost import CatBoostRanker


# CatBoost требует,
# чтобы строки одной группы шли подряд.
ranker_train = (
    ranker_train
    .sort_values(
        [
            "user_id",
            "als_rank"
        ]
    )
    .reset_index(drop=True)
)


ranker = CatBoostRanker(
    iterations=50,
    depth=6,
    learning_rate=0.05,
    loss_function="YetiRank",
    random_seed=42,
    verbose=1
)


ranker.fit(
    ranker_train[FEATURES],
    ranker_train["target"],
    group_id=ranker_train["user_id"]
)

0:	total: 106ms	remaining: 5.19s
1:	total: 153ms	remaining: 3.66s
2:	total: 199ms	remaining: 3.12s
3:	total: 245ms	remaining: 2.81s
4:	total: 289ms	remaining: 2.6s
5:	total: 341ms	remaining: 2.5s
6:	total: 400ms	remaining: 2.46s
7:	total: 462ms	remaining: 2.42s
8:	total: 535ms	remaining: 2.44s
9:	total: 582ms	remaining: 2.33s
10:	total: 631ms	remaining: 2.24s
11:	total: 679ms	remaining: 2.15s
12:	total: 729ms	remaining: 2.08s
13:	total: 786ms	remaining: 2.02s
14:	total: 836ms	remaining: 1.95s
15:	total: 887ms	remaining: 1.88s
16:	total: 931ms	remaining: 1.81s
17:	total: 979ms	remaining: 1.74s
18:	total: 1.03s	remaining: 1.68s
19:	total: 1.08s	remaining: 1.62s
20:	total: 1.12s	remaining: 1.55s
21:	total: 1.17s	remaining: 1.49s
22:	total: 1.21s	remaining: 1.43s
23:	total: 1.26s	remaining: 1.37s
24:	total: 1.31s	remaining: 1.31s
25:	total: 1.35s	remaining: 1.25s
26:	total: 1.41s	remaining: 1.2s
27:	total: 1.46s	remaining: 1.15s
28:	total: 1.51s	remaining: 1.09s
29:	total: 1.55s	remaining:

In [15]:
os.makedirs(
    "models",
    exist_ok=True
)

ranker.save_model(
    "models/catboost_ranker.cbm"
)

In [ ]:
s3.upload_file(
    model_path,
    bucket_name,
    "recsys/models/catboost_ranker.cbm"
)

print("Модель загружена в S3")

In [16]:
# ------------------------------------------------------------
# Получаем итоговый score
# ------------------------------------------------------------

candidates["rank_score"] = (
    ranker
    .predict(
        candidates[FEATURES]
    )
    .astype("float32")
)


recommendations = (
    candidates[
        [
            "user_id",
            "item_id",
            "rank_score"
        ]
    ]
    .sort_values(
        [
            "user_id",
            "rank_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


recommendations["rank"] = (
    recommendations
    .groupby("user_id")
    .cumcount()
    .add(1)
    .astype("int16")
)


recommendations = recommendations.rename(
    columns={
        "rank_score": "score"
    }
)


display(
    recommendations.head(20)
)

,user_id,item_id,score,rank
0,1,91105,1.213419,1
1,1,206880,1.189303,2
2,1,289103,1.105373,3
3,1,274798,1.009995,4
4,1,393028,1.004707,5
5,1,350103,0.935550,6
6,1,11279,0.935550,7
7,1,197642,0.902608,8
8,1,312728,0.726499,9
9,1,45323,0.718760,10


In [17]:
# ------------------------------------------------------------
# Сохраняем итоговые рекомендации
# ------------------------------------------------------------

recommendations.to_parquet(
    "recommendations.parquet",
    index=False
)


# s3.upload_file(
#     "recommendations.parquet",
#     bucket_name,
#     "recsys/recommendations/recommendations.parquet"
# )


print(
    "recommendations.parquet сохранён и загружен в S3"
)

recommendations.parquet сохранён и загружен в S3


In [37]:
%reset -f

import gc
gc.collect()

0